In [2]:
%pwd


'D:\\Medical-Chatbot-Pinecone-AWS\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'D:\\Medical-Chatbot-Pinecone-AWS'

In [5]:
from langchain_community.document_loaders import PyPDFLoader,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [6]:
def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)

    documents=loader.load()

    return documents

In [7]:
extracted_data = load_pdf_file("data")

In [8]:
len(extracted_data)

637

In [9]:
from typing import List
from langchain.schema import Document

In [10]:
def filter_to_minimal_docs(docs: List[Document]) -> List[Document]: 
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [11]:
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [12]:
text_chunks = text_split(extracted_data)

In [28]:
from langchain.embeddings import HuggingFaceEmbeddings
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2',
                                    model_kwargs={"device": "cpu"}) 
    #this model return 384 dimensions
    return embeddings

In [31]:
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
embedding = download_hugging_face_embeddings()


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\rampr\anaconda3\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\rampr\anaconda3\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "C:\Users\rampr\anaconda3\lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\rampr\anaconda3\lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "C:\Users\rampr\anaconda3\lib\site-pack

AttributeError: _ARRAY_API not found

ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.

In [30]:
vector = embedding.embed_query("hello world")
vector

NameError: name 'embedding' is not defined

In [16]:
from dotenv import load_dotenv
import os
load_dotenv()


True

In [17]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
GROQ_API_KEY = os.environ.get('GROQ_API_KEY')

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ['GROQ_API_KEY'] = GROQ_API_KEY 


In [18]:
import sys
!{sys.executable} -m pip install pinecone

In [19]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY
pc =Pinecone(api_key=pinecone_api_key)

In [20]:
from pinecone import ServerlessSpec
index_name = "medical-chatbot" 

In [21]:
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud = "aws" , region ="us-east-1")
    )

In [22]:
index =pc.Index(index_name)

In [23]:
import sys
!{sys.executable} -m pip install langchain-pinecone

In [24]:
from langchain_pinecone import PineconeVectorStore

In [25]:
index_name = "medical-chatbot" 
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embedding
)

NameError: name 'embedding' is not defined

In [26]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

NameError: name 'docsearch' is not defined

In [27]:
retrivede_docs = retriever.invoke("What is acne")

NameError: name 'retriever' is not defined

In [ ]:
retrivede_docs

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade langchain langchain-core langchain-groq langchain-community langchain-pinecone

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name="llama3-8b-8192"  # or "mixtral-8x7b-32768", "gemma-7b-it"
)